# Week 09: More Text

We've seen how to do some basic text manipulation and transformations.

But, what if we have to extract more complex patterns from text, or want to parse structured text files ?

In [1]:
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/data_utils.py

In [2]:
import re
import requests

from bs4 import BeautifulSoup

from data_utils import image_from_url

## Revisit something from WK01

In the very first lecture we briefly saw an example of how to download images by accessing websites through code.

Let's break this down into steps and see how we can use code to "read" websites.

### Requests

The internet is a gigantic, connected, hard drive. When we access an `url` with our browser, what we're actually doing is downloading a file from that location, and interpreting its results as a webpage.

But the file is just an `html` text file. We can download and print it.

In [3]:
res = requests.get("https://www.newschool.edu/")

print(res.text)



<!DOCTYPE html>

<html class="no-js tns centennial whatinput-types-initial gr__events_newschool_edu wf-neue-n9-active wf-neue-n7-active wf-neue-n3-active wf-neue-i3-active wf-neue-n4-active wf-neue-i4-active wf-neuedisplay-n7-active wf-neuedisplay-n4-active wf-neuedisplayrandom-n7-active wf-neuedisplayrandom-n4-active wf-neuedisplaywide-n7-active wf-neuedisplaywide-n4-active wf-neuedisplayultra-n7-active wf-neuedisplayultra-n4-active wf-active" lang="en">
<head><link rel="stylesheet" type="text/css" href="/WorkArea/FrameworkUI/css/ektron.stylesheet.ashx?id=-1759591071+-914034194+-731554408+1064114300+-300771134+1985268503" /><script type="text/javascript" src="/WorkArea/FrameworkUI/js/ektron.javascript.ashx?id=-569449246+-1939951303+-1080527330+-1687560804+-1388997516+2009761168+27274999+1979897163+-422906301+-1818005853+-991739241+-1793043690+1338359439+1743165108+1531089627"></script>
    <script src="https://cdn.optimizely.com/js/20462340719.js"></script>

    <link rel="dns-prefe

## `HTML` / `XML`

`HTML` and `XML` are $2$ formats of structured text files.

They use nested _tags_ (`<tag>content</tag>`) to contextualize or give specific meaning to different parts of the text file.

These tags in `HTML`, for example, are used to specify an image, a link and a list, respectively:

```html
<img src="image.jpg"/>

<a href="https://www.newschool.edu/">Link Text</a>

<ul>
  <li>List Item 1</li>
  <li>List Item 2</li>
</ul>
```

`XML` is a very similar format, but instead of specifying elements to be displayed, it's mainly used for storing and transferring structured text. It's not unlike `JSON`, but focused on text information:

```xml
<document>
  <book>
    <title>Towards a Philosophy of Photography</title>
    <author>Vilém Flusser</author>
    <year>1984</year>
  </book>
</document>
```

In [25]:
# TODO: try other URLs, try to find image tags <img> and link tags <a>

tns_txt = res.text

# for line in tns_txt[:18]:
#   print(line)

for line in tns_txt.split("\n"):
  if "img" in line:
    print(line)

  <img alt="Placa branca com logotipo e texto do Jardim Botânico Pajter Suruí fixada em tronco de árvore em área de mata. Vegetação densa ao fundo iluminada por luzes verdes durante o entardecer." 
  <img alt="Placa branca com logotipo e texto do Jardim Botânico Pajter Suruí fixada em tronco de árvore em área de mata. Vegetação densa ao fundo iluminada por luzes verdes durante o entardecer." 
  <img alt="Osciloscópio com tela mostrando ondas amarelas em padrão repetitivo. Mão ajusta cabos conectados a eletrodos sobre mesa branca em ambiente de laboratório." 
  <img alt="Osciloscópio com tela mostrando ondas amarelas em padrão repetitivo. Mão ajusta cabos conectados a eletrodos sobre mesa branca em ambiente de laboratório." 
  <img alt="Dois homens empurram uma jangada feita de troncos amarrados com cordas na areia da praia. Ao fundo, barco com vela vermelha está ancorado próximo à água." 
  <img alt="Dois homens empurram uma jangada feita de troncos amarrados com cordas na areia da pra

## Why do we care?

Everything is on the internet.

If we know how to automate information retrieval and extraction from urls, we can create our own datasets by carefully processing and combining data from different sources.

For example, maybe we want to create a dataset of news articles about the environment.

We can get a list of recent articles from these newspapers by accessing their RSS feeds (`xml`) or `html` pages:
- https://www.theguardian.com/environment/rss
- https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml
- https://www.latimes.com/environment/rss2.0.xml
- https://www.washingtonpost.com/climate-environment/
- https://www.chinadaily.com.cn/china/environment
- https://timesofindia.indiatimes.com/rssfeeds/2647163.cms
- https://english.elpais.com/climate/
- https://japannews.yomiuri.co.jp/science-nature/climate-change/

Let's try to get a list of article titles, links and images from one of these.

The `split()` and `strip()` functions might be useful, as well as the `in` operator for checking if a specific text is present _in_ a line.

In [ ]:
# TODO: get list of titles, links and images from one of the newspaper feeds

nyt_res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml")
nyt_xml = nyt_res.text.split("\n")

all_titles = []
for line in nyt_xml:
  if "<titles>" in line:
    https_idx = line.find("https")
    jpg_idx = line.find("jpg")
    if https_idx = 
    # a_title = line.replace("<title>", "").replace("</title>", "")
    # print(a_title.strip())

## A good start

Going through an `html` or `xml` file line-by-line is ok. We can quickly get a sense of the document's structure and see how many articles we get, but if we actually want to extract the link, title and image of an article, while keeping that information together... it gets kind of messy.

In [6]:
res = requests.get("https://www1.folha.uol.com.br/ambiente/")
res.encoding = "utf8"
fsp_climate = res.text

# number of characters in the file
print(len(fsp_climate), "characters\n")

# find "c-newslist__title" and "c-most-read__section" to isolate actual list of articles
list0_idx = fsp_climate.find("c-newslist__title")
list1_idx = fsp_climate.find("c-most-read__section")

# only go through the list of articles (skipping headlines 🤦🤷)
fsp_climate_main = fsp_climate[list0_idx:list1_idx]
fsp_climate_lines = fsp_climate_main.split("\n")
print(len(fsp_climate_main), "characters\n")

# use the "class" attribute of specific tags to isolate lines of interest
for l in fsp_climate_lines[:1050]:
  if "c-image-aspect-ratio__link" in l:
    print("  url: ", l.strip())
  elif "c-lazyload" in l:
    print("image: ", l.strip())
  elif "c-headline__title" in l:
    print("title: ", l.strip())

1010672 characters

788655 characters

  url:  <a href="https://www1.folha.uol.com.br/ambiente/2026/04/como-a-ia-ajuda-a-prever-a-florada-das-cerejeiras-no-japao.shtml" class="c-image-aspect-ratio__link">
image:  <img class="c-headline__image c-image-aspect-ratio__image c-lazyload" data-src="https://f.i.uol.com.br/fotografia/2026/04/06/177550025969d3fbe32c8a2_1775500259_3x2_xs.jpg" alt="Visitors gather beneath cherry blossoms at Yoyogi Park in Tokyo, March 24, 2026. (Kentaro Takahashi/The New York Times)">
title:  <h2 class="c-headline__title">Como a IA ajuda a prever a florada das cerejeiras no Japão</h2>
  url:  <a href="https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml" class="c-image-aspect-ratio__link">
image:  <img class="c-headline__image c-image-aspect-ratio__image c-lazyload" data-src="https://f.i.uol.com.br/fotografia/2026/04/01/177506540369cd593b2fa20_1775065403_3x2_xs.jpg" alt="">
title:  <h2 cl

### Try with another link

Try to do something similar to extract article titles, links and images from another newspaper.

The `xml` feeds are a bit easier, but still... this is ***HARD***.

Each page has its own formatting and structuring, and even if we isolate the lines with the content we want, like we did above, we still haven't extracted the titles, images or links, only the `html` or `xml` with that information.

In [7]:
# TODO: try to get titles, links and images from one of the other links

In [ ]:
nyt_res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml")
nyt_xml = nyt_res.text.split("\n")

all_titles = []

for line in 

## RegEx to the RegEx-cue

Instead of trying to come up with specific for loops and parsing logic to find the items we're interested in, we can use _regular expressions_ to describe more sophisticated patterns to extract.

For example, we can find and extract all instances of the pattern `<img`, with:
```
re.findall(r"<img", txt)
```

In [8]:
# match all <img
print(re.findall(r"<img", fsp_climate))

# match all src=
print(re.findall(r"src=", fsp_climate))

['<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img', '<img']
['src='

## 🤔

This is almost useful.

In order to fully unlock all of its powers, we have to learn a few other regular expression commands and modifiers.

### Character Classes
- `.`: Any character except a newline.
- `\d`: Any digit (0-9).
- `\D`: Any non-digit character.
- `\w`: Any word character (alphanumeric plus underscore).
- `\W`: Any non-word character.
- `\s`: Any whitespace character.
- `\S`: Any non-whitespace character.
- `[abc]`: Matches any one character in the set 'a', 'b', or 'c'.
- `[^abc]`: Matches any character not in the set 'a', 'b', or 'c'.
- `[a-z]`: Matches any lowercase letter in the specified range.

### Anchors
- `^`: Matches the start of the string (or start of a line with the re.MULTILINE flag).
- `$`: Matches the end of the string (or end of a line with the re.MULTILINE flag).

### Quantifiers
- `*`: Zero or more repetitions of the preceding element.
- `+`: One or more repetitions of the preceding element.
- `?`: Zero or one repetition of the preceding element.
- `{n}`: Exactly n repetitions.
- `{n,}`: At least n repetitions.
- `{n,m}`: Between n and m repetitions.
- `*?`, `+?`, `??`, `{n,m}?`: Non-greedy versions of the quantifiers, matching as few repetitions as possible.

### Tip

We can help the regex engine and simplify our patterns by removing newlines and extra spaces.

the following code replaces any repeating sequence of whitespace characters with a single space.

In [9]:
fsp_climate_line = re.sub(r"\s+", " ", fsp_climate)

## RegEx for real now

Let's extract clean titles, links and images from the articles.

In [10]:
# match all src= followed by anything followed by a space
# this matches everything between the first "src" in the file until the very last ">" symbol
display(re.findall(r"src.+>", fsp_climate_line)[:4])

# match all src followed by anything until the first ">"
display(re.findall(r"src.+?>", fsp_climate_line)[:4])

# match all "headline content" <div> blocks
display(re.findall(r"<div[^>]+?headline__content.+?</div>", fsp_climate_line)[:4])

['src="https://tm.jsuol.com.br/uoltm.js?id=bt1dcm" defer></script> <link rel="preload" as="script" href="https://tm.jsuol.com.br/modules/external/admanager/folha_ads.js"> <script src="https://tm.jsuol.com.br/modules/external/admanager/folha_ads.js" defer></script> <script> //chartbeat (function(){ var _sf_async_config = window._sf_async_config = (window._sf_async_config || {}); _sf_async_config.uid = 50059; _sf_async_config.cookieDomain = \'folha.uol.com.br\'; _sf_async_config.domain = \'folha.com.br\'; _sf_async_config.topStorageDomain = \'uol.com.br\'; _sf_async_config.flickerControl = false; _sf_async_config.path = \'www1.folha.uol.com.br/ambiente/\'; // _sf_async_config.sections = (_sf_async_config.sections ? _sf_async_config.sections : \'estudio.folha.com.br\'); _sf_async_config.articleBlockSelector = \'div.c-headline\'; var r="anon";try{var u=universal_variable.aud.userType;"logged"==u?r="lgdin":"subscriber"==u&&(r="paid")}catch(a){}var _cbq=window._cbq=window._cbq||[];_cbq.push(

['src="https://tm.jsuol.com.br/uoltm.js?id=bt1dcm" defer>',
 'src="https://tm.jsuol.com.br/modules/external/admanager/folha_ads.js" defer>',
 "src = '//static.chartbeat.com/js/chartbeat.js'; n.parentNode.insertBefore(e, n); } loadChartbeat(); })(); </script>",
 'src="//static.chartbeat.com/js/chartbeat_mab.js">']

['<div class="c-headline__content"> <a href="https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml" class="c-headline__url" > <h2 class="c-headline__title"> Jardim botânico na amazônia reúne etnoturismo, reflorestamento e agrofloresta </h2> <p class="c-headline__standfirst">Projetos sustentáveis guiam o trabalho idealizado pelo povo indígena paiter suruí</p> </a> </div>',
 '<div class="c-headline__content"> <a href="https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml" class="c-headline__url" > <h2 class="c-headline__title"> Microgerador de energia a base de vibração é aposta para reforçar monitoramento ambiental na amazônia </h2> <p class="c-headline__standfirst">Universidade Federal do Amazonas criou protótipo para sensoriamento remoto</p> </a> </div>',
 '<div class="c-headline__content"> <a href="https://www

In [11]:
# use parentheses grouping to match and single out href blocks within the "headline content" <div> blocks
href_pattern = r"<a.+?href=\"(.+?)\""
pattern = fr"<div[^>]+?headline__content.+?{href_pattern}.+?</div>"
display(re.findall(pattern, fsp_climate_line)[:4])

# use 2 parentheses groups to match and single out href and h2 blocks within the "headline content" <div> blocks
h2_pattern = r"<h2.+?>\s*(.+?)\s*</h2>"
link_title_pattern = fr"<div[^>]+?headline__content.+?{href_pattern}.+?{h2_pattern}.+?</div>"
links_titles = re.findall(link_title_pattern, fsp_climate_line)
display(links_titles[:4])

# use parentheses grouping to match and single out img and data-src blocks within the "headline media" <div> blocks
img_pattern = r"<img.+?data-src=\"(.+?)\".+?>"
link_img_pattern = fr"<div[^>]+?headline__media-wrapper.+?{href_pattern}.+?{img_pattern}.+?</div>"
links_imgs = re.findall(link_img_pattern, fsp_climate_line)
display(links_imgs[:4])

['https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml',
 'https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml',
 'https://www1.folha.uol.com.br/ambiente/2026/04/armadilha-biodegradavel-de-lagostas-reduz-danos-ambientais-no-rio-grande-do-norte.shtml',
 'https://arte.folha.uol.com.br/ambiente/2025/grandes-obras-na-amazonia/a-estrada-de-ferro-carajas/trem-da-vale-duplica-altera-radicalmente-vida-de-povo-indigena-de-recente-contato-e-isolados-ficam-no-limbo']

[('https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml',
  'Jardim botânico na amazônia reúne etnoturismo, reflorestamento e agrofloresta'),
 ('https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml',
  'Microgerador de energia a base de vibração é aposta para reforçar monitoramento ambiental na amazônia'),
 ('https://www1.folha.uol.com.br/ambiente/2026/04/armadilha-biodegradavel-de-lagostas-reduz-danos-ambientais-no-rio-grande-do-norte.shtml',
  'Armadilha biodegradável de lagostas reduz danos ambientais no Rio Grande do Norte'),
 ('https://arte.folha.uol.com.br/ambiente/2025/grandes-obras-na-amazonia/a-estrada-de-ferro-carajas/trem-da-vale-duplica-altera-radicalmente-vida-de-povo-indigena-de-recente-contato-e-isolados-ficam-no-limbo',
  'Trem da Vale altera radicalmente vida de povo indígena de recente contat

[('https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml',
  'https://f.i.uol.com.br/fotografia/2026/04/01/177506540369cd593b2fa20_1775065403_3x2_sm.jpg'),
 ('https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml',
  'https://f.i.uol.com.br/fotografia/2026/02/20/17716295466998ebea20b46_1771629546_3x2_sm.jpg'),
 ('https://www1.folha.uol.com.br/ambiente/2026/04/armadilha-biodegradavel-de-lagostas-reduz-danos-ambientais-no-rio-grande-do-norte.shtml',
  'https://f.i.uol.com.br/fotografia/2026/03/18/177386718469bb10b00b8f7_1773867184_3x2_sm.jpg'),
 ('https://arte.folha.uol.com.br/ambiente/2025/grandes-obras-na-amazonia/a-estrada-de-ferro-carajas/trem-da-vale-duplica-altera-radicalmente-vida-de-povo-indigena-de-recente-contato-e-isolados-ficam-no-limbo',
  'https://f.i.uol.com.br/fotografia/2026/01/12/176826216869658a

### Clean up

We have links and titles in one list, and links and images in another.

We can use the links to make sure all our articles have all three properties.

In [12]:
link2info = { link: { "link": link, "title": title } for link, title in links_titles }

for link,img in links_imgs:
  if link in link2info:
    link2info[link]["img"] = img
  else:
    raise(Exception(f"missing: {link}"))

list(link2info.values())[:4]

[{'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml',
  'title': 'Jardim botânico na amazônia reúne etnoturismo, agrofloresta e reflorestamento',
  'img': 'https://f.i.uol.com.br/fotografia/2026/04/01/177506540369cd593b2fa20_1775065403_3x2_xs.jpg'},
 {'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml',
  'title': 'Microgerador de energia a base de vibração é aposta para reforçar pesquisa ambiental na amazônia',
  'img': 'https://f.i.uol.com.br/fotografia/2026/02/20/17716295466998ebea20b46_1771629546_3x2_xs.jpg'},
 {'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/armadilha-biodegradavel-de-lagostas-reduz-danos-ambientais-no-rio-grande-do-norte.shtml',
  'title': 'Armadilha biodegradável de lagostas reduz danos ambientais no Rio Grande do Norte',
  'img': 'https://f.i.uol.com.br/f

## Exercise

Use RegEx patterns to get urls and download images from a google/bing image search.

The `image_from_url()` function imported above can be used to turn an image `url` into a `PIL` image.

### The Search

The following are urls for doing image searches on google and bing:

In [13]:
QUERY_TERM = "gudetama"

BING_URL = f"https://www.bing.com/images/search?q={QUERY_TERM}"
GOOGLE_URL = f"https://www.google.com/search?udm=2&q={QUERY_TERM}"

In [14]:
# TODO: Extract image urls and display images

## `HTML`/`XML` Selectors

RegEx is good and efficient, but can get a little confusing at times.

Depending on the structure of the `html`/`xml` file we're trying to parse, it might be useful to use a library that can traverse the file using tag names, attributes and hierarchical structure.

For example, consider the following `html`/`xml` file structures:

```xml
<tag attribute="value" id="unique-name" class="list of classes">
  content
</tag>
```

```xml
<tag attribute="value" id="unique-name" class="list of classes">
  outer content
  <child attribute="value" id="unique-name" class="list of classes">
    inner content
  </child>
  more content
</tag>
```

We can use `HTML`/`XML` _selectors_ to directly filter for elements with particular properties.

### Selectors

- `tag`: select by tag name
- `#id`: select by id
- `.class`: select by class

- `[attribute]`: select all elements with an attribute
- `[attribute='value']`: select all elements with an attribute with a given value

- `sel1,sel2`: select all sel1 and all sel2 elements
- `sel1 sel2`: select all sel2 elements that are children of sel1
- `sel1 > sel2`: select all sel2 elements that are direct children of sel1

## Beautiful Soup

The `Beautiful Soup` library gives us a pretty easy way to do exactly this.

We just have to give it a `html` or `xml` string, and then we can use the `select()` function to traverse the file elements.

In [15]:
res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/US.xml")
nyt_txt = res.text
nyt_soup = BeautifulSoup(nyt_txt, features="xml")

### Get all `<item>` tags

In [16]:
for item in nyt_soup.select("item"):
  print(item)

<item>
<title>Hegseth Likens Easter Rescue of U.S. Airman to Resurrection of Jesus Christ</title>
<link>https://www.nytimes.com/2026/04/06/us/politics/hegseth-religious-tone.html</link>
<guid isPermaLink="true">https://www.nytimes.com/2026/04/06/us/politics/hegseth-religious-tone.html</guid>
<atom:link href="https://www.nytimes.com/2026/04/06/us/politics/hegseth-religious-tone.html" rel="standout"/>
<description>President Trump also asserted that God supports the American war against Iran “because God is good, and God wants to see people taken care of.”</description>
<dc:creator>Chris Cameron</dc:creator>
<pubDate>Mon, 06 Apr 2026 23:38:58 +0000</pubDate>
<category domain="http://www.nytimes.com/namespaces/keywords/nyt_per">Hegseth, Pete</category>
<category domain="http://www.nytimes.com/namespaces/keywords/des">US and Israeli Attack on Iran (2026)</category>
<category domain="http://www.nytimes.com/namespaces/keywords/des">Easter and Holy Week</category>
<media:content height="1800" 

### Get all `<title>` tags that are children of `<item>` tags

In [17]:
for title in nyt_soup.select("item title"):
  print(title)

<title>Hegseth Likens Easter Rescue of U.S. Airman to Resurrection of Jesus Christ</title>
<title>Trump Says Iran Cease-Fire Proposal Is ‘Not Good Enough’ as Deadline Approaches</title>
<title>A Redistricting War in Florida, Georgia’s Runoff and the Latest Politics News</title>
<title>6 Takeaways From Trump’s News Conference on Iran</title>
<title>Texas Congressman Accused of Pursuing a Second Subordinate With Lewd Texts</title>
<title>Between Easter Eggs and Bunny Hops, Trump Talks of War and Autopens</title>
<title>Shots Fired at Indianapolis Councilman’s Home, After Vote Backing Data Center</title>
<title>Trump Administration Pulls Out of Civil Rights Settlements Backing Trans Students</title>
<title>Where does the data for election estimates come from, and what kind of data do we collect?</title>
<title>Do You Have Questions About a No-Bid Federal Contract? Tell Us Here.</title>
<title>The California Lake Billed as the ‘Saudi Arabia of Lithium’</title>
<title>Supreme Court Clears t

### Get all `<item>` tags, then get its `<link>`, `<title>` and `<media>` children

This: 
```py
item.select("media|content[medium='image']")
```

selects every `<media:content>` tag inside `item` that has a `medium` attribute set to `image`.

Something like this:

```xml
<item>
  ...
  <media:content width="800" height="450" medium="image" url="https://static01.nyt.com/images/file/jpg" />
  ...
</item>
```

In [18]:
for item in nyt_soup.select("item"):
  title = item.select("title")
  link = item.select("link")
  media = item.select("media|content[medium='image']")
  if len(media) > 0:
    print(f"{title[0].text}\n {link[0].text}\n {media[0]['url']}\n")

Hegseth Likens Easter Rescue of U.S. Airman to Resurrection of Jesus Christ
 https://www.nytimes.com/2026/04/06/us/politics/hegseth-religious-tone.html
 https://static01.nyt.com/images/2026/04/06/multimedia/06israel-iran-religious-tone2-zthv/06israel-iran-religious-tone2-zthv-mediumSquareAt3X.jpg

Trump Says Iran Cease-Fire Proposal Is ‘Not Good Enough’ as Deadline Approaches
 https://www.nytimes.com/2026/04/06/us/politics/trump-iran-cease-fire-proposal.html
 https://static01.nyt.com/images/2026/04/06/multimedia/06dc-diplo-mvcw/06dc-diplo-mvcw-mediumSquareAt3X.jpg

A Redistricting War in Florida, Georgia’s Runoff and the Latest Politics News
 https://www.nytimes.com/2026/04/06/us/politics/redistricting-virginia-florida-ga-special-election-republicans.html
 https://static01.nyt.com/images/2026/04/06/us/politics/newsletter-2/newsletter-2-mediumSquareAt3X.jpg

6 Takeaways From Trump’s News Conference on Iran
 https://www.nytimes.com/2026/04/06/us/politics/trump-iran-news-conference.html
 

### With `html`

We can go directly to the elements that have the content we are interested in.

In the case of the `https://www1.folha.uol.com.br/ambiente/` page, we're looking for:
- all of the `<div>` elements that belong to the `.c-headline` class

<br>Inside of those:
- the title is in an `<h2>` tag inside a `<div>` of class `.c-headline__content`
- the article url is in an `<a>` tag inside a `<div>` of class `.c-headline__content`
- the image url is in an `<img>` tag inside a `<div>` of class `.c-headline__media-wrapper`

In [19]:
res = requests.get("https://www1.folha.uol.com.br/ambiente/")
res.encoding = "utf8"
fsp_climate = res.text
fsp_soup = BeautifulSoup(fsp_climate, "html.parser")

In [20]:
fsp_headlines = fsp_soup.select(".c-headline")

articles = []

for headline in fsp_headlines:
  title_el = headline.select(".c-headline__content h2")
  link_el = headline.select(".c-headline__content a")
  img_el = headline.select(".c-headline__media-wrapper img")
  if len(title_el) > 0:
    articles.append({
      "title": title_el[0].text.strip(),
      "link": link_el[0]['href'],
      "image": img_el[0]['data-src'],
    })

display(len(articles))
display(articles[:8])

106

[{'title': 'Jardim botânico na amazônia reúne etnoturismo, reflorestamento e agrofloresta',
  'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/jardim-botanico-na-amazonia-reune-etnoturismo-reflorestamento-e-agrofloresta.shtml',
  'image': 'https://f.i.uol.com.br/fotografia/2026/04/01/177506540369cd593b2fa20_1775065403_3x2_sm.jpg'},
 {'title': 'Microgerador de energia a base de vibração é aposta para reforçar monitoramento ambiental na amazônia',
  'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/microgerador-de-energia-a-base-de-vibracao-e-aposta-para-reforcar-monitoramento-ambiental-na-amazonia.shtml',
  'image': 'https://f.i.uol.com.br/fotografia/2026/02/20/17716295466998ebea20b46_1771629546_3x2_sm.jpg'},
 {'title': 'Armadilha biodegradável de lagostas reduz danos ambientais no Rio Grande do Norte',
  'link': 'https://www1.folha.uol.com.br/ambiente/2026/04/armadilha-biodegradavel-de-lagostas-reduz-danos-ambientais-no-rio-grande-do-norte.shtml',
  'image': 'https://f.i.u

### Yeah !

Much easier than plain RegEx for deeply nested `html`/`xml` files.